## Part 1: Distributed Data Processing with Spark

In [4]:
#ALL IMPORTED LIBRARIES USED
import requests
import os 
import time

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window

### Task 1.1: Spark Environment Setup & Data Loading

#### Creating the Spark Session Locally

In [5]:
# Set JAVA_HOME to the Microsoft Java that's actually working
os.environ['JAVA_HOME'] = r'C:\Program Files\Microsoft\jdk-11.0.16.101-hotspot'
os.environ['PATH'] = r'C:\Program Files\Microsoft\jdk-11.0.16.101-hotspot\bin;' + os.environ['PATH']
#The above code isn't needed and my environment variables just need fixing, if this is still here i probally forgot
#to remove it sorry


spark = SparkSession.builder \
 .master('local[*]') \
 .appName('COMP3610_Assignment3') \
 .config('spark.sql.adaptive.enabled', 'true') \
 .config('spark.driver.memory', '4g') \
 .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
 .getOrCreate()

# Verify the session
print(f'Spark version: {spark.version}')
print(f'App name: {spark.sparkContext.appName}')
print(f'Master: {spark.sparkContext.master}')
print(f'Default parallelism: {spark.sparkContext.defaultParallelism}')

Spark version: 3.3.2
App name: COMP3610_Assignment3
Master: local[*]
Default parallelism: 8


In [6]:
# The Spark UI is available at http://localhost:4040 while the session is active
print(f'Spark UI: http://localhost:4040')
print(f'Number of executor cores: {spark.sparkContext.defaultParallelism}')
# List available configurations
for key, value in sorted(spark.sparkContext.getConf().getAll()):
 if 'memory' in key.lower() or 'core' in key.lower() or 'master' in key.lower():
    print(f' {key} = {value}')

Spark UI: http://localhost:4040
Number of executor cores: 8
 spark.driver.memory = 4g
 spark.master = local[*]


####  Loading the NYC Yellow Taxi Parquet data into a Spark DataFrame and Pandas DataFrame

In [7]:
#Download NYC Taxi Data Locally First

#This is the directory where the files are stored
os.makedirs("data/raw", exist_ok=True)
YELLOW_TAXI_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
DESTINATION_PATH = "data/raw/yellow_tripdata_2024-01.parquet"


print(f"Beginning download of {YELLOW_TAXI_URL}...")
try:
    #Make request to url for the parquet file
    response = requests.get(YELLOW_TAXI_URL, stream=True)
    response.raise_for_status() # Check if request was successful

    #We then open the file to download the data to
    with open(DESTINATION_PATH, 'wb') as file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                file.write(chunk)

    print(f"Download completed and saved to {DESTINATION_PATH}.")
    print(f"Total file size is : {os.path.getsize(DESTINATION_PATH) / 1e6:.1f} MB")
except requests.exceptions.RequestException as e:
    print(f"An error occurred while downloading {YELLOW_TAXI_URL}: {e}")

Beginning download of https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet...
Download completed and saved to data/raw/yellow_tripdata_2024-01.parquet.
Total file size is : 50.0 MB


In [8]:
# Load the data into a Spark DataFrame and Check Load Time
pyspark_time_start = time.time()
pyspark_df = spark.read.parquet(DESTINATION_PATH)
pyspark_load_time = time.time() - pyspark_time_start

# Load the data into a Pandas DataFrame and Check Load Time
pandas_time_start = time.time()
pandas_df = pd.read_parquet(DESTINATION_PATH)
pandas_load_time = time.time() - pandas_time_start

print(f"pyspark load time: {pyspark_load_time: .2f}s")
print(f"pandas load time: {pandas_load_time : .2f}s")

pyspark load time:  5.32s
pandas load time:  1.31s


We notice substantial time increase when loading into a Pandas DataFrame compared to a Pyspark DataFrame since Pyspark leverages distributed and parallel processing across a cluster of machines whereas pandas is limited to in-memory, single-machine processing. Furthermore Pyspark is Multi-threaded while Pandas is Single-threaded

However on first load we do notice a substantial time increase of 3-4 seconds due to the cold start up overheads and lazy evaluation setup

#### Reporting Details of Dataset using Pyspark DataFrame

In [9]:
pyspark_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [10]:
print(f"The number of rows in the dataset are : {pyspark_df.count()}")
print(f'Partition count : {pyspark_df.rdd.getNumPartitions()} \n\n')

print("The dataset is as follows:")
pyspark_df.show(5, truncate=True)


The number of rows in the dataset are : 2964624
Partition count : 8 


The dataset is as follows:
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2023-12-31 20:57:55|  2023-12-31 21:17:43|              1|         1.72|      

### Task 1.2: Data Cleaning & Feature Engineering in Spark

#### Removing rows with null values in: pickup/dropoff times, locations, fare, distance

In [11]:
CRITICAL_COLS = [
    'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'PULocationID', 'DOLocationID', 'fare_amount', 'trip_distance',
]

rows_before_null_filter = pyspark_df.count()
cleaned_pyspark_df = pyspark_df.dropna(subset=CRITICAL_COLS)

print(f"The initial rows before null filtering were:        {rows_before_null_filter}")
print(f"The rows after the null filter in the dataset is:   {cleaned_pyspark_df.count()}")
print(f"Total rows removed were:                            {rows_before_null_filter-pyspark_df.count()}")

The initial rows before null filtering were:        2964624
The rows after the null filter in the dataset is:   2964624
Total rows removed were:                            0


#### Filtering out invalid trips: zero/negative distance, negative fares, fares exceeding $500, dropoff before pickup

In [12]:
rows_before_invalid_filter = pyspark_df.count()
pyspark_df = pyspark_df.filter(
    (F.col('trip_distance') > 0) &
    (F.col('fare_amount') >= 0) &
    (F.col('fare_amount') <= 500) &
    (F.col('tpep_dropoff_datetime') > F.col('tpep_pickup_datetime')) 
)

print(f"The initial rows before filtering out invalid columns were:         {rows_before_invalid_filter}")
print(f"The rows after filtering out invalid columns in the dataset is:     {pyspark_df.count()}")
print(f"Total rows removed were:                                            {rows_before_invalid_filter-pyspark_df.count()}")

The initial rows before filtering out invalid columns were:         2964624
The rows after filtering out invalid columns in the dataset is:     2870046
Total rows removed were:                                            94578


#### Creating derived columns for better data analysis

In [13]:
#Adding trip_duration_minutes column
pyspark_df = pyspark_df.withColumn(
    'trip_duration_minutes',
    ((F.unix_timestamp('tpep_dropoff_datetime') -
    F.unix_timestamp('tpep_pickup_datetime')) / 60)
    .cast(DoubleType())
)

#Adding trip_speed_mph column
pyspark_df = pyspark_df.withColumn(
    'trip_speed_mph',
    (F.when(F.col('trip_duration_minutes')/60 > 0,
           F.col('trip_distance') / (F.col('trip_duration_minutes') / 60.0))
    .otherwise(None))
    .cast(DoubleType())
)

#Adding pickup_hour column
pyspark_df = pyspark_df.withColumn(
    'pickup_hour',
    F.hour('tpep_pickup_datetime')
)

#Adding pickup_day_of_week column
pyspark_df = pyspark_df.withColumn(
    'pickup_day_of_week',
    F.dayofweek('tpep_pickup_datetime')
)

#Adding tip_percentage column
pyspark_df = pyspark_df.withColumn(
    'tip_percentage',
    (F.when((F.col('fare_amount') >0),
           F.col('tip_amount')/F.col('fare_amount') * 100))
    .otherwise(None)
    .cast(DoubleType())
)

### Task 1.3 — Spark SQL Analytics

#### Query 1: What are the top 10 busiest pickup hours, and what is the average fare and tip percentage for each?

In [14]:
# Register as a temporary view
pyspark_df.createOrReplaceTempView('taxi_trips')

In [15]:
spark.sql("""
    SELECT
        ROW_NUMBER() OVER (ORDER BY trip_count DESC) AS rank,
        pickup_hour,
        trip_count,
        avg_fare,
        avg_tip_pct
    FROM (
        SELECT
            pickup_hour,
            COUNT(*) AS trip_count,
            ROUND(AVG(fare_amount), 2) AS avg_fare,
            ROUND(AVG(tip_percentage), 2) AS avg_tip_pct
        FROM taxi_trips
        GROUP BY pickup_hour
    )
    ORDER BY rank
    LIMIT 10
""").show()

+----+-----------+----------+--------+-----------+
|rank|pickup_hour|trip_count|avg_fare|avg_tip_pct|
+----+-----------+----------+--------+-----------+
|   1|         14|    206281|   17.01|      22.78|
|   2|         13|    200310|   18.12|      22.34|
|   3|         12|    184968|   19.46|      21.84|
|   4|         11|    184004|   19.11|       19.8|
|   5|         15|    178810|   17.63|      22.86|
|   6|         10|    178026|   19.27|       19.8|
|   7|          9|    165355|   18.42|      19.79|
|   8|          8|    159912|    17.8|      19.74|
|   9|         17|    155910|   18.29|      21.88|
|  10|         16|    155559|   18.05|      22.17|
+----+-----------+----------+--------+-----------+



We notice that the busiest pickup hours are in the afternoon from 12-2pm, this may be due to people leaving work or going out for leisure activities. Without the day of the week to compare it can be generalized that most people use the taxi service after work hours to go home or in the morning from 8-10am to arrive to work.

#### Query 2: Which day of the week has the highest average trip speed? Include average distance and duration

In [24]:
spark.sql(""" 
    SELECT 
          pickup_day_of_week,
          CASE pickup_day_of_week
            WHEN 1 THEN 'Sunday'    WHEN 2 THEN 'Monday'
            WHEN 3 THEN 'Tuesday'   WHEN 4 THEN 'Wednesday'
            WHEN 5 THEN 'Thursday'  WHEN 6 THEN 'Friday'
            WHEN 7 THEN 'Saturday'
          END  AS day_name,
          ROUND(AVG(trip_speed_mph), 2) AS avg_trip_speed,
          ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
          ROUND(AVG(trip_duration_minutes), 2) AS avg_trip_duration
    FROM taxi_trips
    WHERE trip_speed_mph IS NOT NULL
    GROUP BY pickup_day_of_week
    ORDER BY avg_trip_speed DESC
""").show()

+------------------+---------+--------------+-----------------+-----------------+
|pickup_day_of_week| day_name|avg_trip_speed|avg_trip_distance|avg_trip_duration|
+------------------+---------+--------------+-----------------+-----------------+
|                 3|  Tuesday|         17.44|             4.24|            16.16|
|                 1|   Sunday|         16.15|             4.07|             14.7|
|                 2|   Monday|         13.98|             3.83|             15.8|
|                 7| Saturday|         13.34|             3.38|            14.91|
|                 6|   Friday|         13.18|             3.58|            15.72|
|                 5| Thursday|         12.58|             3.53|            16.37|
|                 4|Wednesday|         12.36|             3.59|            16.23|
+------------------+---------+--------------+-----------------+-----------------+



We notice that Tuesday shows the highest trip speed along with Sunday, this may be due to reduced traffic congestion on these days so the taxi's can move faster with the trip distance and duration also being longer due to this.

#### Query 3: Using a window function, rank the top 5 pickup locations by total revenue for each day of the week.

In [39]:
spark.sql("""
        WITH revenue_by_loc AS(
          SELECT
                pickup_day_of_week,
                PULocationID as pickup_location,
                ROUND(SUM(total_amount),2) as total_revenue
          FROM taxi_trips
          GROUP BY pickup_day_of_week, PULocationID
        ),
        ranked AS(
          SELECT *,
                RANK() OVER(
                        PARTITION BY pickup_day_of_week
                        ORDER BY total_revenue DESC
                ) AS revenue_rank
          FROM revenue_by_loc
        )
        SELECT * FROM ranked
        WHERE revenue_rank <=5
        ORDER BY revenue_rank, pickup_day_of_week     
""").show(35, truncate=False)

+------------------+---------------+-------------+------------+
|pickup_day_of_week|pickup_location|total_revenue|revenue_rank|
+------------------+---------------+-------------+------------+
|1                 |132            |1611212.28   |1           |
|2                 |132            |2068017.7    |1           |
|3                 |132            |1786490.1    |1           |
|4                 |132            |1635524.25   |1           |
|5                 |132            |1417026.64   |1           |
|6                 |132            |1384194.47   |1           |
|7                 |132            |1283109.29   |1           |
|1                 |138            |789210.77    |2           |
|2                 |138            |1015948.07   |2           |
|3                 |138            |946888.41    |2           |
|4                 |138            |991222.8     |2           |
|5                 |138            |844295.29    |2           |
|6                 |138            |7629

Without the zone table it is hard to know exactly where these locations are however it can be safe to assume that the areas which generate the highest revenue are airports and city centers where commuters travel into the city or around it to get to work, home or for leisure activities. It should also be noted that weekdays generate the most amount of money which may be from commuters going to and from work

#### Query 4: Calculate the cumulative trip count by hour of day (running total from hour 0 to 23). At what hour does the cumulative count surpass 50% of daily trips?

In [ ]:
query4 = spark.sql("""
        WITH hourly AS (
            SELECT pickup_hour, COUNT(*) AS hourly_trips
            FROM taxi_trips
            GROUP BY pickup_hour
        ),
        cumulative AS(
            SELECT pickup_hour, hourly_trips,
            SUM(hourly_trips) OVER (
                ORDER BY pickup_hour
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS cumulative_trips
            FROM hourly
        ),
        total AS (SELECT SUM(hourly_trips) AS total_trips FROM hourly )
        SELECT pickup_hour, hourly_trips, cumulative_trips,
               ROUND(cumulative_trips/total_trips * 100,2) as cumulative_trips_perc
        FROM cumulative CROSS JOIN total
        ORDER BY pickup_hour
           
""")

query4.show(24)
crossover_hour = query4.filter(query4.cumulative_trips_perc >= 50).first().pickup_hour
print(f"The hour at which the cumulative count surpasses 50% of daily trips is hour {crossover_hour}")

+-----------+------------+----------------+---------------------+
|pickup_hour|hourly_trips|cumulative_trips|cumulative_trips_perc|
+-----------+------------+----------------+---------------------+
|          0|       15284|           15284|                 0.53|
|          1|       17495|           32779|                 1.14|
|          2|       39415|           72194|                 2.52|
|          3|       80870|          153064|                 5.33|
|          4|      113506|          266570|                 9.29|
|          5|      125619|          392189|                13.66|
|          6|      135425|          527614|                18.38|
|          7|      146754|          674368|                 23.5|
|          8|      159912|          834280|                29.07|
|          9|      165355|          999635|                34.83|
|         10|      178026|         1177661|                41.03|
|         11|      184004|         1361665|                47.44|
|         

The cumulative count for trips surpasses 50% of the total daily trips at around midday 12pm indicating that half of the trips in the day are for commuters going to work and the other half are for them going from from work in the afternoon with the taxi service

#### Query 5: Compare average fare, distance, and tip percentage between short trips (<2miles), medium trips (2–10 miles), and long trips (>10 miles). Which category has the highest tip percentage?


In [69]:
query5 = spark.sql("""
        SELECT
            CASE
                WHEN trip_distance < 2 THEN 'Short trip (< 2  miles)'
                WHEN trip_distance <= 10 THEN 'Medium trip (2-10 miles)'
                ELSE 'Long trip (> 10 miles)'
            END AS distance_category,
            ROUND(AVG(fare_amount), 2) AS avg_fare,
            ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
            ROUND(AVG(tip_percentage), 2) AS avg_tip_pct
        FROM taxi_trips
        GROUP BY distance_category
""")

query5.show()

max_tip_perc = query5.orderBy(F.desc('avg_tip_pct')).first()

print(f"The trip category with the highest tip percentage is {max_tip_perc.distance_category} with a tip percentage of {max_tip_perc.avg_tip_pct}%")

+--------------------+--------+-----------------+-----------+
|   distance_category|avg_fare|avg_trip_distance|avg_tip_pct|
+--------------------+--------+-----------------+-----------+
|Medium trip (2-10...|   22.18|             3.96|      18.57|
|Short trip (< 2  ...|    9.91|             1.13|      23.07|
|Long trip (> 10 m...|   64.65|             21.7|      21.93|
+--------------------+--------+-----------------+-----------+

The trip category with the highest tip percentage is Short trip (< 2  miles) with a tip percentage of 23.07%


We observe that for longer trips the average fare is siginicantly greater but shor trips still bring in the higher tip percentages as customers are more pleased with a shorter trip than one that is longer, however medium trips received the lowest tip percentage with customers being more likely to tip for shorter trips or ones which are long.

### Task 1.4: Performance Optimization 